Customers Table

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_customers = 500

customers = pd.DataFrame({
    "customer_id": range(1, n_customers + 1),
    "age": np.random.randint(18, 70, n_customers),
    "gender": np.random.choice(['Male', 'Female'], n_customers),
    "city": ['Larisa']*n_customers,
    "registration_date": pd.to_datetime("2023-01-01")+
                         pd.to_timedelta(np.random.randint(0, 365, n_customers), unit='D')
})

customers.head()

,customer_id,age,gender,city,registration_date
0,1,56,Female,Larisa,2023-02-20
1,2,69,Male,Larisa,2023-03-22
2,3,46,Male,Larisa,2023-05-13
3,4,32,Female,Larisa,2023-10-12
4,5,60,Male,Larisa,2023-05-18


In [5]:
def assign_segment(age):
    if age < 30:
        return "Young"
    elif age < 50:
        return "Adult"
    else:
        return "Senior"

customers["segment"]=customers["age"].apply(assign_segment)

customers.head()

,customer_id,age,gender,city,registration_date,segment
0,1,56,Female,Larisa,2023-02-20,Senior
1,2,69,Male,Larisa,2023-03-22,Senior
2,3,46,Male,Larisa,2023-05-13,Adult
3,4,32,Female,Larisa,2023-10-12,Adult
4,5,60,Male,Larisa,2023-05-18,Senior


Products Table

In [6]:
products = pd.DataFrame({
    "product_id": range(1,16),
    "product_name": [
        "Milk", "Bread", "Feta Cheese", "Yogurt", "Olive Oil",
        "Pasta", "Rice", "Chicken", "Eggs", "Apples",
        "Coffee", "Sugar", "Butter", "Juice", "Soft Drinks"
    ],
    "category": [
        "Dairy", "Bakery", "Dairy", "Dairy", "Grocery",
        "Grocery", "Grocery", "Meat", "Dairy", "Fruits",
        "Beverages", "Grocery", "Dairy", "Beverages", "Beverages"
    ]
})

In [7]:
# add pricing

np.random.seed(42)

products["cost_price"] = np.round(np.random.uniform(0.5, 10, len(products)), 2)

# add margin (20%-60%)
products["selling_price"] = np.round(
    products["cost_price"] * np.random.uniform(1.2, 1.6, len(products)), 2)

In [8]:
products

,product_id,product_name,category,cost_price,selling_price
0,1,Milk,Dairy,4.06,5.17
1,2,Bread,Bakery,9.53,12.60
2,3,Feta Cheese,Dairy,7.45,10.50
3,4,Yogurt,Dairy,6.19,8.50
4,5,Olive Oil,Grocery,1.98,2.61
5,6,Pasta,Grocery,1.98,2.86
6,7,Rice,Grocery,1.05,1.32
7,8,Chicken,Meat,8.73,11.50
8,9,Eggs,Dairy,6.21,8.36
9,10,Apples,Fruits,7.23,9.99


Transactions Table

In [10]:
np.random.seed(42)

n_transactions = 2000

transactions = pd.DataFrame({
    "transaction_id": range(1, n_transactions + 1),
    "customer_id": np.random.choice(customers["customer_id"], n_transactions),
    "transaction_date": pd.to_datetime("2024-01-01") + 
                        pd.to_timedelta(np.random.randint(0, 365, n_transactions), unit='D')
})

transactions.head()

,transaction_id,customer_id,transaction_date
0,1,103,2024-06-07
1,2,436,2024-05-31
2,3,349,2024-04-17
3,4,271,2024-07-06
4,5,107,2024-10-27


Transaction Items Table

In [12]:
np.random.seed(42)

transaction_items = []

for t_id in transactions["transaction_id"]:
    n_items = np.random.randint(1, 6)

    for _ in range(n_items):
        product = np.random.choice(products["product_id"])
        qty = np.random.randint(1, 4)

        transaction_items.append([t_id, product, qty])

transaction_items = pd.DataFrame(transaction_items, columns=[
    "transaction_id", "product_id", "quantity"
])

transaction_items.head()

,transaction_id,product_id,quantity
0,1,13,3
1,1,11,1
2,1,5,3
3,1,10,3
4,2,11,1


Full Sales Table

In [13]:
sales = transaction_items.merge(products, on="product_id")
sales = sales.merge(transactions, on="transaction_id")

sales["revenue"] = sales["quantity"] * sales["selling_price"]

sales.head()

,transaction_id,product_id,quantity,product_name,category,cost_price,selling_price,customer_id,transaction_date,revenue
0,1,13,3,Butter,Dairy,8.41,11.82,103,2024-06-07,35.46
1,1,11,1,Coffee,Beverages,0.70,1.06,103,2024-06-07,1.06
2,1,5,3,Olive Oil,Grocery,1.98,2.61,103,2024-06-07,7.83
3,1,10,3,Apples,Fruits,7.23,9.99,103,2024-06-07,29.97
4,2,11,1,Coffee,Beverages,0.70,1.06,436,2024-05-31,1.06


✔ Customer behavior dataset

✔ Product catalog with margins

✔ Transaction system

✔ Revenue calculation layer

KPI ENGINE

Total Revenue

In [14]:
total_revenue = sales["revenue"].sum()
total_revenue

np.float64(84558.34)

Average Order Value (AOV)

In [15]:
aov = sales.groupby("transaction_id")["revenue"].sum().mean()
aov

np.float64(42.27917)

Total Transactions

In [16]:
total_transactions = transactions["transaction_id"].unique()
total_transactions

array([   1,    2,    3, ..., 1998, 1999, 2000])

Customer KPIs (VERY important for consulting)

🔹 Revenue per Customer

In [17]:
customer_revenue = sales.groupby("customer_id")["revenue"].sum()
customer_revenue.head()

customer_id
1    300.14
2    134.59
3      3.18
4    201.88
5    397.90
Name: revenue, dtype: float64

🔹 Top Customers (VIPs)

In [19]:
top_customers = customer_revenue.sort_values(ascending=False).head(10)
top_customers

customer_id
476    561.65
268    555.79
435    484.00
417    472.10
264    459.19
39     446.62
469    430.69
342    427.51
441    425.42
147    424.28
Name: revenue, dtype: float64

🔹 Customer Activity

In [20]:
active_customers = sales["customer_id"].nunique()
active_customers

489

Product KPIs (this is where businesses make decisions)

🔹 Best Selling Products

In [21]:
top_products = sales.groupby("product_name")["revenue"].sum().sort_values(ascending=False)
top_products

product_name
Sugar          10503.35
Butter          9976.08
Bread           9676.80
Chicken         8901.00
Apples          8501.49
Feta Cheese     7791.00
Yogurt          7123.00
Eggs            6570.96
Milk            4316.95
Juice           2874.28
Pasta           2356.64
Olive Oil       2106.27
Soft Drinks     2078.08
Rice            1036.20
Coffee           746.24
Name: revenue, dtype: float64

🔹 Quantity Sold per Product

In [22]:
product_volume = sales.groupby("product_name")["quantity"].sum().sort_values(ascending=False)
product_volume

product_name
Apples         851
Sugar          845
Butter         844
Yogurt         838
Milk           835
Pasta          824
Olive Oil      807
Juice          794
Eggs           786
Rice           785
Chicken        774
Bread          768
Soft Drinks    764
Feta Cheese    742
Coffee         704
Name: quantity, dtype: int64

🔹 Profit per Product (IMPORTANT)

In [23]:
sales["profit"] = (sales["selling_price"] - sales["cost_price"]) * sales["quantity"]

product_profit = sales.groupby("product_name")["profit"].sum().sort_values(ascending=False)
product_profit

product_name
Butter         2878.04
Bread          2357.76
Apples         2348.76
Sugar          2298.40
Feta Cheese    2263.10
Chicken        2143.98
Yogurt         1935.78
Eggs           1689.90
Milk            926.85
Juice           873.40
Pasta           725.12
Olive Oil       508.41
Soft Drinks     374.36
Coffee          253.44
Rice            211.95
Name: profit, dtype: float64

Time-Based KPIs (VERY impressive in dashboards)

🔹 Monthly Revenue Trend

In [24]:
sales['month'] = sales["transaction_date"].dt.to_period("M")

monthly_revenue = sales.groupby("month")["revenue"].sum()
monthly_revenue

month
2024-01    7219.39
2024-02    6340.23
2024-03    7370.27
2024-04    6926.11
2024-05    8265.42
2024-06    7636.62
2024-07    6330.38
2024-08    7059.86
2024-09    6784.54
2024-10    7262.01
2024-11    6220.99
2024-12    7142.52
Freq: M, Name: revenue, dtype: float64

Business Insight Example:

“Your top 20% of products generate most of your revenue, while 30% are barely profitable.”

“Your average customer spends X€, but VIP customers spend 3–4x more.”

“Revenue peaks in specific months, meaning inventory planning can be optimized.”

In [25]:
sales.to_csv("sales.csv", index=False)